In [1]:
from typing import Optional
import string
from dataclasses import dataclass
from tqdm import tqdm

import numpy as np
import pandas as pd

import hnswlib
import faiss

import torch
from torch import nn

from torch.utils.data import Dataset, DataLoader

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

from transformers import AutoTokenizer, AutoModel


In [2]:
dataset = pd.read_parquet("products_with_names.parquet")

In [3]:
@dataclass
class Document:
    document_id: int
    document_name: str

documents = [
    Document(document_id=doc[1]["product_id"], document_name=doc[1]["name"]) 
    for doc in dataset.iterrows() 
    if doc[1]["name"] != ""
]


In [4]:
documents[:5]

[Document(document_id=4036767, document_name='Модуль сменный фильтрующий Аквафор КН, 208731'),
 Document(document_id=4050873, document_name='Водоочиститель Аквафор модель Кристалл Н, 205963 //с краном'),
 Document(document_id=4226160, document_name='Развиваем мышление (2-3 года) | Земцова Ольга'),
 Document(document_id=4644911, document_name='Lacoste Вода парфюмерная Pour Femme 50 мл'),
 Document(document_id=4788809, document_name='Сменные Кассеты Для Мужской Бритвы Gillette Mach3, с 3 лезвиями, прочнее, чем сталь, для точного бритья, 2 шт')]

In [5]:
class SimpleTextProcessor:
    def __init__(self):
        self.symbols_to_replace = {"ё": "е"}

    def lowercase_text(self, text: str) -> str:
        return text.lower()

    def replace_symbols(self, text: str) -> str:
        for old, new in self.symbols_to_replace.items():
            text = text.replace(old, new)
        return text

    def process_punctuation_simple(self, text: str) -> str:
        translation_table = str.maketrans(string.punctuation, ' ' * len(string.punctuation))
        text_without_punc = text.translate(translation_table)
        text_without_double_spaces = ' '.join(text_without_punc.split())
        return text_without_double_spaces

    def process_text(self, text: str) -> str:
        text = self.lowercase_text(text)
        text = self.replace_symbols(text)
        text = self.process_punctuation_simple(text)
        return text
        

In [6]:
text_processor = SimpleTextProcessor()

In [7]:
documents_processed = [
    Document(document.document_id, text_processor.process_text(document.document_name)) 
    for document in tqdm(documents)
]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 238428/238428 [00:03<00:00, 76552.35it/s]


In [8]:
documents_processed[:5]

[Document(document_id=4036767, document_name='модуль сменный фильтрующий аквафор кн 208731'),
 Document(document_id=4050873, document_name='водоочиститель аквафор модель кристалл н 205963 с краном'),
 Document(document_id=4226160, document_name='развиваем мышление 2 3 года земцова ольга'),
 Document(document_id=4644911, document_name='lacoste вода парфюмерная pour femme 50 мл'),
 Document(document_id=4788809, document_name='сменные кассеты для мужской бритвы gillette mach3 с 3 лезвиями прочнее чем сталь для точного бритья 2 шт')]

## Tokenization

In [9]:
corpus = [doc.document_name for doc in documents_processed]

In [10]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"],
    vocab_size=10_000,
    min_frequency=2,
    show_progress=True,
    continuing_subword_prefix="##",
)

tokenizer.train_from_iterator(corpus, trainer)

tokenizer.save("bpe_tokenizer.json")

In [11]:
tokenizer = Tokenizer.from_file("bpe_tokenizer.json")

In [12]:
encoding = tokenizer.encode("мороженое для собак")
print(f"Tokens: {encoding.tokens}")
print(f"Token ids: {encoding.ids}")

Tokens: ['мороженое', 'для', 'собак']
Token ids: [2326, 195, 566]


In [13]:
encoding = tokenizer.encode("бритва gillette")
print(f"Tokens: {encoding.tokens}")
print(f"Token ids: {encoding.ids}")

Tokens: ['бритва', 'gillette']
Token ids: [3905, 4069]


In [14]:
encoding = tokenizer.encode("шампунь мужской nivea men")
print(f"Tokens: {encoding.tokens}")
print(f"Token ids: {encoding.ids}")

Tokens: ['шампунь', 'мужской', 'nivea', 'men']
Token ids: [815, 1278, 3090, 2877]


In [15]:
encoding = tokenizer.encode("шампунь мужской nivea men охлаждающий")
print(f"Tokens: {encoding.tokens}")
print(f"Token ids: {encoding.ids}")

Tokens: ['шампунь', 'мужской', 'nivea', 'men', 'охлажда', '##ющий']
Token ids: [815, 1278, 3090, 2877, 7492, 300]


In [17]:
tokenizer.decode(encoding.ids).replace(" ##", "")

'шампунь мужской nivea men охлаждающий'

In [18]:
VOCAB_SIZE = tokenizer.get_vocab_size()
print(f"Tokenizer vocab size: {VOCAB_SIZE}")

Tokenizer vocab size: 10000


## Building DSSM

In [19]:
class DSSM(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int = 256, hidden_dims: list[int] = [512, 256, 128], padding_idx: int = 3):
        super().__init__()

        self.padding_idx = padding_idx

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)

        layers = []
        input_dim = embedding_dim
        for dim in hidden_dims:
            layers.append(nn.LayerNorm(input_dim))
            layers.append(nn.Linear(input_dim, dim))
            layers.append(nn.LeakyReLU())
            input_dim = dim

        self.mlp = nn.Sequential(*layers)

        self.ln = nn.LayerNorm(input_dim)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
        
    def forward(self, input_ids: torch.LongTensor):
        input_embeddings = self.embedding(input_ids)  # [batch_size, seq_len, embedding_dim]

        input_embeddings_pooled = (
            torch.sum(input_embeddings, dim=1) / torch.sum(input_ids != self.padding_idx, dim=1, keepdim=True)  # [batch_size, embedding_dim]
        )

        projected = self.mlp(input_embeddings_pooled)  # [batch_size, output_dim]

        return self.ln(projected)
    


## Prepare data for training

In [20]:
query_product_positive_interactions = pd.read_parquet("query_product_positive_interactions.parquet")

In [21]:
query_product_positive_interactions.shape

(18121972, 4)

In [22]:
query_product_positive_interactions.head()

,user_id,timestamp,search_query,product_name
0,9897711,2024-03-16 11:28:10,линзы,"ACUVUE Контактные линзы, -3.75, 8.4, 2 недели"
1,3666669,2024-04-15 13:40:39,линзы -4,"ACUVUE Контактные линзы, -3.75, 8.4, 2 недели"
2,4951147,2024-04-28 06:23:20,линзы acuvue,"ACUVUE Контактные линзы, -3.75, 8.4, 2 недели"
3,972605,2024-04-25 13:00:18,линзы acuvue,"ACUVUE Контактные линзы, -3.75, 8.4, 2 недели"
4,972605,2024-03-10 07:08:38,линзы acuvue,"ACUVUE Контактные линзы, -3.75, 8.4, 2 недели"


In [24]:
data = []

In [25]:
for row in tqdm(query_product_positive_interactions.head(500_000).iterrows()):
    data.append(
        (
            text_processor.process_text(row[1].search_query), 
            text_processor.process_text(row[1].product_name),
        )
    )

500000it [00:32, 15522.16it/s]


In [26]:
data[-5:]

[('пюре фруктовое',
  'детское фруктово ягодное пюре агуша яблоко банан клубника киви с 8 месяцев 90г'),
 ('фруктовое пюре',
  'детское фруктово ягодное пюре агуша яблоко банан клубника киви с 8 месяцев 90г'),
 ('пюре фруктовое детское',
  'детское фруктово ягодное пюре агуша яблоко банан клубника киви с 8 месяцев 90г'),
 ('детское питание пюре',
  'детское фруктово ягодное пюре агуша яблоко банан клубника киви с 8 месяцев 90г'),
 ('бананы',
  'детское фруктово ягодное пюре агуша яблоко банан клубника киви с 8 месяцев 90г')]

In [27]:
class QueryDocDataset(Dataset):
    def __init__(self, data: list[tuple[str, str]], tokenizer: Tokenizer, max_len: int = 32):
        self.queries = [row[0] for row in data]
        self.docs = [row[1] for row in data]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.queries)

    def __getitem__(self, idx: int):
        query = self.queries[idx]
        doc = self.docs[idx]
        
        query_ids = self.tokenizer.encode(query).ids[:self.max_len]
        doc_ids = self.tokenizer.encode(doc).ids[:self.max_len]
        
        return {
            "query_ids": torch.LongTensor(query_ids),
            "doc_ids": torch.LongTensor(doc_ids),
        }

def collate_fn(batch: list[dict[str, torch.LongTensor]]):
    query_ids = torch.nn.utils.rnn.pad_sequence(
        [item["query_ids"] for item in batch], batch_first=True, padding_value=3
    )
    doc_ids = torch.nn.utils.rnn.pad_sequence(
        [item["doc_ids"] for item in batch], batch_first=True, padding_value=3
    )
    return {"query_ids": query_ids, "doc_ids": doc_ids}


In [28]:
DEVICE = "cuda:1" if torch.cuda.is_available() else None

In [30]:
dataset = QueryDocDataset(data, tokenizer)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, collate_fn=collate_fn)
model = DSSM(vocab_size=tokenizer.get_vocab_size())
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

model.to(DEVICE)

def train_step(batch: dict[str, torch.LongTensor]):

    optimizer.zero_grad()

    query_ids, doc_ids = batch["query_ids"], batch["doc_ids"]

    query_embs = model(query_ids.to(DEVICE))
    doc_embs = model(doc_ids.to(DEVICE))

    query_embs = torch.nn.functional.normalize(query_embs, p=2, dim=1)
    doc_embs = torch.nn.functional.normalize(doc_embs, p=2, dim=1)

    scores = torch.matmul(query_embs, doc_embs.T)  # [batch_size, batch_size]

    labels = torch.arange(len(scores), device=scores.device)

    loss = loss_fn(scores, labels)

    loss.backward()
    optimizer.step()

    return loss.item()

for epoch in range(10):
    train_losses = []
    for i, batch in enumerate(dataloader):
        loss = train_step(batch)
        train_losses.append(loss)
        if i % 1000 == 0:
            print(f"Epoch: {epoch}, iteration: {i}, Loss: {sum(train_losses) / len(train_losses):.4f}")
            train_losses = []

Epoch: 0, iteration: 0, Loss: 4.6563
Epoch: 0, iteration: 1000, Loss: 3.9823
Epoch: 0, iteration: 2000, Loss: 3.9352
Epoch: 0, iteration: 3000, Loss: 3.9290
Epoch: 1, iteration: 0, Loss: 3.9141
Epoch: 1, iteration: 1000, Loss: 3.9217
Epoch: 1, iteration: 2000, Loss: 3.9220


KeyboardInterrupt: 

In [31]:
def cosine_similarity(vec_1: torch.FloatTensor, vec_2: torch.FloatTensor):
    vec_1_normalized = torch.nn.functional.normalize(vec_1, p=2, dim=1)
    vec_2_normalized = torch.nn.functional.normalize(vec_2, p=2, dim=1)
    scores = torch.sum(vec_1_normalized * vec_2_normalized, dim=1)

    return scores.item()

In [40]:
query = "линзы"
doc = "acuvue"

In [41]:
query_embedding = model(torch.LongTensor([tokenizer.encode(query).ids]).to(DEVICE))

In [42]:
document_embedding = model(torch.LongTensor([tokenizer.encode(doc).ids]).to(DEVICE))

In [43]:
cosine_similarity(query_embedding, document_embedding)

0.9995273351669312

## ANN index

In [44]:
model.eval()

DSSM(
  (embedding): Embedding(10000, 256, padding_idx=3)
  (mlp): Sequential(
    (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=256, out_features=512, bias=True)
    (2): LeakyReLU(negative_slope=0.01)
    (3): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
    (7): Linear(in_features=256, out_features=128, bias=True)
    (8): LeakyReLU(negative_slope=0.01)
  )
  (ln): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
)

In [45]:
document_names = [x.document_name for x in documents_processed]

In [46]:
document_names[:3]

['модуль сменный фильтрующий аквафор кн 208731',
 'водоочиститель аквафор модель кристалл н 205963 с краном',
 'развиваем мышление 2 3 года земцова ольга']

In [47]:
def embed_texts(texts: list[str], model: nn.Module, tokenizer: Tokenizer, max_len: int = 32):
    model.eval()
    embeddings = []
    with torch.no_grad():
        for text in tqdm(texts):
            ids = tokenizer.encode(text).ids[:max_len]
            ids = torch.LongTensor(ids).unsqueeze(0).to(DEVICE)
            emb = model(ids)
            embeddings.append(emb.squeeze(0).cpu().numpy())
    return np.stack(embeddings)

doc_embeddings = embed_texts(document_names, model, tokenizer)
print(f"Document embeddings shape: {doc_embeddings.shape}") 

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 238428/238428 [01:39<00:00, 2400.48it/s]


Document embeddings shape: (238428, 128)


In [48]:
dim = doc_embeddings.shape[1]
index = hnswlib.Index(space="cosine", dim=dim)

index.init_index(max_elements=len(documents), ef_construction=200, M=16)
index.add_items(doc_embeddings)

In [49]:
query = "сыр сливочный"
query_embedding = embed_texts([query], model, tokenizer)[0]

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 710.90it/s]


In [50]:
k = 5
labels, distances = index.knn_query(query_embedding, k=k)

for i, (label, dist) in enumerate(zip(labels[0], distances[0])):
    print(f"{documents[label]} (Score: {1 - dist})")

Document(document_id=142583583, document_name='Сыр плавленый Сливочный 50% 400 г, Viola') (Score: 0.9999889135360718)
Document(document_id=146123911, document_name='Сыр Сливочный 45% 150 г, Natura, нарезка') (Score: 0.9999874830245972)
Document(document_id=149171034, document_name='Сыр плавленый с грибами 55% 400 г, Hochland') (Score: 0.999981701374054)
Document(document_id=149171037, document_name='Сыр творожный с зеленью 60% 150 г, Almette') (Score: 0.9999814033508301)
Document(document_id=1119531410, document_name='Сыр творожный воздушный с зеленью 60% 150 г, SVEZA') (Score: 0.9999799728393555)


In [52]:
class DSSMRetriever:
    def __init__(
        self, 
        model: nn.Module, 
        tokenizer: Tokenizer, 
        documents: list[Document], 
        document_names: list[str], 
        doc_embeddings: Optional[torch.Tensor] = None,
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.documents = documents
        self.document_names = document_names
        
        self.dim = 128
        self.index = hnswlib.Index(space="cosine", dim=self.dim)
        self.index.init_index(max_elements=len(self.document_names), ef_construction=200, M=16)
        
        doc_embeddings = embed_texts(self.document_names, model, tokenizer) if doc_embeddings is None else doc_embeddings
        self.index.add_items(doc_embeddings)
    
    def search(self, query, k=5):
        query_embedding = embed_texts([query], self.model, self.tokenizer)[0]
        labels, _ = self.index.knn_query(query_embedding, k=k)
        return [self.documents[label] for label in labels[0]]

retriever = DSSMRetriever(model, tokenizer, documents, document_names, doc_embeddings)


In [53]:
results = retriever.search("корм для собак", k=10)

results

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 703.62it/s]


[Document(document_id=27524214, document_name='Сухой корм Pedigree для взрослых собак маленьких пород, с говядиной, 2.2кг'),
 Document(document_id=175412944, document_name='Сухой корм для стерилизованных собак Royal Canin Mini Sterilised для мелких пород, 3 кг'),
 Document(document_id=27914674, document_name='Сухой корм Pedigree для взрослых собак маленьких пород, с говядиной, 600г'),
 Document(document_id=144695883, document_name='Сухой корм для собак Eukanuba для породы лабрадор ретривер, с курицей, 10 кг'),
 Document(document_id=141838268, document_name='Сухой корм для собак мелких пород Royal Canin Indoor, 3 кг'),
 Document(document_id=258529657, document_name='Влажный корм Pedigree для собак миниатюрных пород, паштет с курицей, 80г'),
 Document(document_id=499415379, document_name='Влажный корм Whiskas для кошек (желе), с лососем, 75 г'),
 Document(document_id=1516963365, document_name='Сухой корм для собак всех пород Родные Корма 1 пуд, с курицей, 16,38 кг. Уцененный товар'),
 Do

## ColBERT

In [54]:
class ColBERT:
    def __init__(self, model_path="./rubert-tiny2/"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModel.from_pretrained(model_path).to(DEVICE)
        self.dim = self.model.config.hidden_size
        self.model.eval()
        
    def encode(self, texts: list[str]):
        inputs = self.tokenizer(
            texts,
            padding=True,
            max_length=64,
            truncation=True,
            return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            outputs = self.model(**inputs)
        return outputs.last_hidden_state.cpu()  # [batch_size, seq_length, dim]
    
    def compute_scores(self, query_emb, doc_emb):
        """MaxSim aggregation"""
        query_embs = torch.nn.functional.normalize(query_emb, p=2, dim=-1)
        doc_embs = torch.nn.functional.normalize(doc_emb, p=2, dim=-1)

        sims = torch.matmul(query_embs, doc_embs.transpose(-2, -1))  # [batch_size, query_length, document_length]
        return torch.max(sims, dim=-1).values.sum(dim=-1)


In [55]:
class FaissIndex:
    def __init__(self, dim: int):
        self.index = faiss.IndexFlatIP(dim)
        
    def add_docs(self, embeddings):
        """Add mean-pooled document embeddings"""
        doc_vectors = embeddings.mean(dim=1).numpy()
        self.index.add(doc_vectors)
        
    def search(self, query_emb, k=5):
        query_vector = query_emb.mean(dim=1).numpy()
        distances, labels = self.index.search(query_vector, k=k)
        return labels, distances


In [56]:
colbert = ColBERT()
index = FaissIndex(dim=colbert.dim)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./rubert-tiny2/
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [57]:
document_names = [x.document_name for x in documents]

In [58]:
doc_embeddings = []
for i in tqdm(range(0, 240_000, 10_000)):
    doc_embeddings.append(colbert.encode(document_names[i:i+10000]))


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 24/24 [00:39<00:00,  1.65s/it]


In [64]:
doc_embeddings[0].shape

torch.Size([64, 312])

In [59]:
doc_embeddings = torch.cat(doc_embeddings, dim=0)

In [60]:
index.add_docs(doc_embeddings)

In [61]:
query = "сливочный сыр"
query_embedding = colbert.encode([query])

In [62]:
labels, scores = index.search(query_embedding, k=5)
    
for i, (label, score) in enumerate(zip(labels[0], scores[0])):
    print(f"{documents[label]} (Score: {score:.3f})")

Document(document_id=366603655, document_name='Пряник имбирный') (Score: 105.740)
Document(document_id=1318951337, document_name='Молочный шоколад СЧАСТЬЕ Kids') (Score: 104.892)
Document(document_id=144576577, document_name='Сыр Сливочный плавленый 45% 400 г, President') (Score: 104.349)
Document(document_id=473287134, document_name='Соус Calve Горчичный с медом, 230 г ') (Score: 104.320)
Document(document_id=1601557446, document_name='Перец красный сладкий, 400 г') (Score: 104.217)


### Search over ColBERT embeddings

In [67]:
class ColBERTSearcher:
    def __init__(self, colbert_model: ColBERT):
        self.model = colbert_model
        self.doc_embeddings = []
        self.documents = []
        
    def index(self, documents: list[str], batch_size: int = 8_192):
        self.documents = documents

        print("Start indexing")
        for i in range(0, len(documents), batch_size):
            print(f"processed docs: {i} / {len(documents)}")
            batch_docs = documents[i:i+batch_size]
            embs = self.model.encode(batch_docs)
            
            for emb in embs:
                self.doc_embeddings.append(emb)
        

    def search(self, query: str, top_k: int = 5, score_batch_size: int = 64):
        q_emb = self.model.encode([query]).to(DEVICE)

        all_scores = []

        for i in range(0, len(self.doc_embeddings), score_batch_size):
            batch_embs = torch.stack(self.doc_embeddings[i:i+score_batch_size]).to(DEVICE)

            scores = self.model.compute_scores(q_emb, batch_embs)

            all_scores.append(scores.cpu().squeeze(0))
            del batch_embs, scores

        all_scores = torch.cat(all_scores)

        top_indices = torch.topk(all_scores, k=top_k).indices.tolist()

        return [
            {"doc": self.documents[idx], "score": all_scores[idx].item()}
            for idx in top_indices
        ]


In [68]:
colbert = ColBERT(model_path="./rubert-tiny2/")
searcher = ColBERTSearcher(colbert)

searcher.index(document_names)


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ./rubert-tiny2/
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Start indexing
processed docs: 0 / 238428
processed docs: 8192 / 238428
processed docs: 16384 / 238428
processed docs: 24576 / 238428
processed docs: 32768 / 238428
processed docs: 40960 / 238428
processed docs: 49152 / 238428
processed docs: 57344 / 238428
processed docs: 65536 / 238428
processed docs: 73728 / 238428
processed docs: 81920 / 238428
processed docs: 90112 / 238428
processed docs: 98304 / 238428
processed docs: 106496 / 238428
processed docs: 114688 / 238428
processed docs: 122880 / 238428
processed docs: 131072 / 238428
processed docs: 139264 / 238428
processed docs: 147456 / 238428
processed docs: 155648 / 238428
processed docs: 163840 / 238428
processed docs: 172032 / 238428
processed docs: 180224 / 238428
processed docs: 188416 / 238428
processed docs: 196608 / 238428
processed docs: 204800 / 238428
processed docs: 212992 / 238428
processed docs: 221184 / 238428
processed docs: 229376 / 238428
processed docs: 237568 / 238428


In [69]:
query = "сыр сливочный"
results = searcher.search(query, top_k=10)

for i, res in enumerate(results):
    print(f"Doc: '{res['doc']}' Score: {res['score']:.4f}")

Doc: 'Сыр творожный SVEZA сливочный, 150 г' Score: 5.0503
Doc: 'Сыр плавленый сливочный 50% 140 г, Hochland, порционный' Score: 5.0434
Doc: 'Сыр Сливочный плавленый 45% 140 г, President' Score: 5.0429
Doc: 'Сыр Сливочный плавленый 45% 200 г, President' Score: 5.0410
Doc: 'Сыр Сливочный плавленый 45% 400 г, President' Score: 5.0303
Doc: 'Сыр Сливочный Лёгкий 16% 200 г, Natura' Score: 5.0282
Doc: 'Сыр Сливочный Лёгкий 16% 400 г, Natura' Score: 5.0271
Doc: 'Сыр плавленый сливочный, с грибами в сливочном соусе 50%, 140 г, Hochland, порционный' Score: 5.0133
Doc: 'Сыр плавленый сливочный 50%, 140 г, Hochland, порционный' Score: 5.0040
Doc: 'Сыр Сливочный Лёгкий 30%, 180 г, Natura' Score: 5.0019


In [70]:
query = "корм для собак"
results = searcher.search(query, top_k=10)

for i, res in enumerate(results):
    print(f"Doc: '{res['doc']}' Score: {res['score']:.4f}")

Doc: 'Игрушка для собак морковка' Score: 4.4703
Doc: 'Beeztees когтерез для собак' Score: 4.1756
Doc: 'Когтерезка для кошек и собак' Score: 4.1472
Doc: 'Когтерезка для кошек и собак' Score: 4.1472
Doc: 'Кормушка для птиц' Score: 4.0865
Doc: 'Капли для животных. Для ушей ТОП-ВЕТ для собак и кошек, 2 шт.' Score: 4.0849
Doc: 'Игрушка для собак HAGEN Носорог, 28 см' Score: 4.0746
Doc: 'Игрушка для собак HAGEN Лев, 23 см' Score: 4.0656
Doc: 'Адресник для собак и кошек на ошейник' Score: 4.0646
Doc: 'Игрушка для собак HAGEN Носорог, 23 см' Score: 4.0568


In [71]:
query = "кефир"
results = searcher.search(query, top_k=10)

for i, res in enumerate(results):
    print(f"Doc: '{res['doc']}' Score: {res['score']:.4f}")

Doc: 'Букет Зефир' Score: 3.3043
Doc: 'Кефир детский 200 г, Обнимама' Score: 3.1153
Doc: 'Кефир "Из Углича" 1%, 500 г' Score: 3.1086
Doc: 'Кефир 1% 900 мл Кавказский долгожитель' Score: 3.1075
Doc: 'Кефир 2,5% 900 г, Сернур' Score: 3.0847
Doc: 'Кефир 2,5% 900 г, МСК-Волжский тетра пак' Score: 3.0737
Doc: 'Кефир обезжиренный 930 г, Вкусняев' Score: 3.0667
Doc: 'Кефир 2,5% 450 мл' Score: 3.0572
Doc: 'Кефир Чабан Халяль, 1%, 450 г' Score: 3.0557
Doc: 'Кефир 1% 900 г, Кубанский Молочник' Score: 3.0520
